# Ejercicios básicos: Hugging Face `transformers`

**Semana 10 · NLP** — Notebook único con 10 ejercicios introductorios.

**Antes: qué significa cada término clave**

- **`transformers` (librería):** Colección de código para cargar modelos de lenguaje ya entrenados, sus tokenizadores y utilidades. No es “un modelo”, es la **herramienta** que los usa.
- **`torch` (PyTorch):** Librería numérica; aquí sirve para **tensores** (arrays multidimensionales) y para ejecutar el modelo en CPU o GPU.
- **Tokenizador:** Función/objeto que **parte el texto en trozos** (tokens) y los convierte en **números** (`input_ids`) según un vocabulario fijo.
- **`AutoTokenizer` / `AutoModel`:** Clases que **eligen solas** qué implementación usar según el nombre del modelo que indiques (p. ej. BERT).
- **`pipeline`:** Función de alto nivel que encadena tokenizador + modelo + postproceso para una **tarea con nombre** (sentimiento, rellenar máscaras, etc.).
- **Decodificar:** Pasar de **números otra vez a texto** legible (`decode`, ver tokens sueltos).

**Objetivo:** Familiarizarte con la API de `transformers`: tokenizadores, modelos `Auto*`, `pipeline` y utilidades de decodificación.

**Requisitos sugeridos:** Python 3.9+, `torch`, `transformers`. Si algo falla al importar, ejecuta la celda de instalación siguiente.

---

## Entorno (antes de los ejercicios)

**Antes: qué significa cada cosa**

- **`pip`:** Gestor de paquetes de Python; **descarga e instala** librerías desde internet en tu entorno actual.
- **Paquete / módulo:** Carpeta de código importable (p. ej. `transformers`). Si no está instalado, `import transformers` falla.
- **Entorno / intérprete:** El **Python concreto** que ejecuta el notebook. Instalar con “otro” Python deja el notebook sin ver el paquete; por eso la celda usa `sys.executable -m pip`.
- **`torch`:** Nombre del paquete de **PyTorch** (tensores y red neuronal).
- **`device` (CPU/GPU):** Dónde viven los cálculos: **CPU** siempre disponible; **GPU (CUDA)** más rápida si existe y está bien configurada.

**Qué hace esta celda:** Comprueba si ya tienes `torch` y `transformers`; si falta alguno, lo instala con `pip` usando **el mismo Python** que ejecuta el notebook (así no se mezclan entornos).

**Ejemplo:** En un ordenador nuevo, la primera vez puede tardar un poco en descargar paquetes; las siguientes ejecuciones solo importan.

**Sintaxis típica (si prefieres instalar tú desde la terminal):**
```bash
pip install transformers torch
```

In [1]:
# Herramientas para comprobar si un paquete ya está instalado y para invocar pip
import importlib.util
import subprocess
import sys


def _ensure(pkg: str) -> None:
    # find_spec devuelve None si el módulo no existe en el entorno actual
    if importlib.util.find_spec(pkg) is None:
        # Instala el paquete con el mismo intérprete que ejecuta este notebook (-m pip)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])


# Garantiza PyTorch y Hugging Face Transformers antes de importarlos
_ensure("torch")
_ensure("transformers")

# Librería de modelos/tokenizers y tensores
import transformers
import torch

# Usa GPU si hay CUDA; si no, CPU (más lento pero siempre disponible)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Comprueba versiones (útil para reproducir resultados o depurar)
print("transformers:", transformers.__version__)
print("torch:", torch.__version__)
print("device:", device)

transformers: 5.5.4
torch: 2.11.0
device: cpu


---
## Ejercicio 1 — Comprobar la versión y el import

**Antes: qué significa cada cosa**

- **`import`:** Ordena a Python: “carga este paquete en memoria para usarlo en el código”.
- **`transformers.__version__`:** Atributo de texto con el **número de versión** instalado (cambia cuando actualizas la librería).
- **Versión:** Importante porque la **API** (nombres de funciones, avisos) puede variar entre versiones mayores.

**Qué practicas:** Verificar que `transformers` está instalado y **anotar la versión exacta**. Eso te permite buscar la documentación adecuada y describir bien un error (por ejemplo: “me falla con transformers 5.5.4”).

**Ejemplo:** Si la salida es `5.5.4`, cualquier guía que diga “compatible con la rama 5.x” debería aplicar a tu entorno.

**Sintaxis:**
```python
import transformers
transformers.__version__
```

**Tarea:** Imprime la versión de `transformers` y confirma que el import no lanza error.

In [2]:
# Alias corto para el paquete principal de Hugging Face
import transformers as tr

# Muestra la versión instalada (debe coincidir con la documentación que consultes)
print(tr.__version__)

5.5.4


---
## Ejercicio 2 — Cargar un tokenizador con `AutoTokenizer`

**Antes: qué significa cada cosa**

- **`AutoTokenizer`:** Clase que **detecta automáticamente** qué tokenizador corresponde al nombre del modelo (no tienes que escribir `BertTokenizer` a mano si no quieres).
- **`from_pretrained("...")`:** Descarga (la primera vez) o **carga desde caché** los archivos del modelo/tokenizer desde el Hub o disco.
- **`bert-base-uncased`:** Nombre de un modelo BERT “base”; **uncased** = ignora mayúsculas al tokenizar.
- **`input_ids`:** Lista o tensor de **enteros**, uno por trozo de texto (token); es la entrada numérica del modelo.
- **Vocabulario:** Tabla fija **palabra/trozo → número**; el tokenizador la usa para traducir texto ↔ IDs.

**Qué practicas:** Obtener el **tokenizador** ligado a un modelo concreto. Su trabajo es **convertir texto en números** (`input_ids`) y preparar **máscaras** (qué posiciones son texto real). `AutoTokenizer` elige solo la clase adecuada (BERT, GPT, etc.) según el nombre que pongas en `from_pretrained`.

**Ejemplo mínimo:** Una frase como `"Hola mundo"` puede pasar a una lista de enteros `[101, 7592, 2088, 102]` (los números reales dependen del vocabulario). Esa lista son los `input_ids`.

**Sintaxis:**
```python
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("nombre-del-modelo")
```

**Tarea:** Carga `bert-base-uncased` y guarda el objeto en `tokenizer`. Muestra el tipo de objeto con `type(tokenizer)`.

In [3]:
# AutoTokenizer descarga (si hace falta) y carga el tokenizador asociado al modelo
from transformers import AutoTokenizer

# bert-base-uncased: BERT en inglés, sin distinguir mayúsculas/minúsculas
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
# Confirma la clase concreta (p. ej. BertTokenizer o BertTokenizerFast)
print(type(tokenizer))

<class 'transformers.models.bert.tokenization_bert.BertTokenizer'>


---
## Ejercicio 3 — Tokenizar una sola frase (salida en listas de Python)

**Antes: qué significa cada cosa**

- **Tokenizar:** Pasar de texto bruto a **secuencia de tokens** (y a sus IDs).
- **`input_ids`:** Identificador numérico de cada token (incluye símbolos especiales como inicio/fin de frase).
- **`attention_mask`:** Lista paralela de **0 y 1**: `1` = “aquí hay token real”; `0` = “padding” (hueco artificial al alinear frases).
- **`token_type_ids` (BERT):** Indica **segmento A o B** cuando metes dos partes en la misma entrada (p. ej. pregunta + párrafo); en una sola frase suele ser todo ceros.
- **Diccionario (`dict`):** Estructura `clave → valor`; aquí cada clave es un tensor de entrada distinto para el modelo.

**Qué practicas:** Llamar al tokenizador como si fuera una función: `tokenizer(texto)`. Obtienes un **diccionario** con listas de Python (aún **no** tensores). Lo habitual es ver `input_ids` (números por token), `attention_mask` (1 = token real, 0 = relleno) y, en BERT, `token_type_ids` (útil cuando hay dos segmentos, p. ej. pregunta y párrafo).

**Ejemplo mínimo:** Si imprimes las claves, algo como `['input_ids', 'token_type_ids', 'attention_mask']` te confirma qué campos trae tu modelo.

**Sintaxis:**
```python
out = tokenizer("Tu texto aquí")
out["input_ids"]
out["attention_mask"]
```

**Tarea:** Tokeniza la frase `"Transformers makes NLP easier"` e imprime las claves del diccionario y la longitud de `input_ids`.

In [4]:
# Frase de ejemplo en inglés (mismo vocabulario que bert-base-uncased)
texto = "Transformers makes NLP easier"
# Tokeniza: devuelve un dict con IDs, máscara de atención, etc.
out = tokenizer(texto)
# Nombres de los campos que el tokenizador rellenó
print(list(out.keys()))
# Número de tokens (incluye [CLS], [SEP] y subpalabras si aplica)
print("len(input_ids):", len(out["input_ids"]))

['input_ids', 'token_type_ids', 'attention_mask']
len(input_ids): 7


---
## Ejercicio 4 — Obtener tensores listos para el modelo (`return_tensors`)

**Antes: qué significa cada cosa**

- **Tensor:** Array numérico multidimensional en PyTorch (similar a un `ndarray`); es lo que el modelo **multiplica y transforma** por capas.
- **`return_tensors="pt"`:** Pide salida en formato **PyTorch** (`pt`); también existen `"tf"` (TensorFlow) o `"np"` (NumPy) según el stack.
- **`batch`:** Aquí, diccionario de tensores con **todas las entradas** de una o varias frases ya alineadas.
- **`device`:** CPU o GPU donde deben estar **tensores y modelo** para poder operar juntos.
- **`shape`:** Forma del tensor, p. ej. `[1, 7]` = 1 secuencia de 7 tokens.

**Qué practicas:** El modelo (`AutoModel`) trabaja con **tensores** de PyTorch, no con listas sueltas. Con `return_tensors="pt"` el tokenizador devuelve `torch.Tensor` listos para pasar a `model(...)`.

**Ejemplo mínimo:** Sin `return_tensors` tienes algo como `[101, 2023, 102]` (lista). Con `return_tensors="pt"` obtienes un tensor de forma `[1, 3]`: una fila = un texto en el mini-lote, cada columna = un token.

**Sintaxis:**
```python
batch = tokenizer("texto", return_tensors="pt")
batch["input_ids"].shape  # [batch, seq_len]
```

**Tarea:** Tokeniza la misma frase del ejercicio 3 con `return_tensors="pt"`, mueve los tensores a `device` y muestra `shape` de `input_ids`.

In [5]:
# Misma frase que en el ejercicio anterior
texto = "Transformers makes NLP easier"
# return_tensors="pt" convierte listas en tensores de PyTorch listos para el modelo
batch = tokenizer(texto, return_tensors="pt")
# Mueve cada tensor al dispositivo (CPU o GPU) para que coincida con el modelo
batch = {k: v.to(device) for k, v in batch.items()}
# Forma [tamaño_lote, longitud_secuencia]; aquí el lote es 1
print(batch["input_ids"].shape)

torch.Size([1, 7])


---
## Ejercicio 5 — Cargar `AutoModel` y obtener el último estado oculto

**Antes: qué significa cada cosa**

- **`AutoModel`:** Carga el **tronco** del modelo (en BERT: *stack* de capas transformer), **sin** la cabeza de clasificación específica de una tarea.
- **Forward / paso hacia adelante:** Evaluar el modelo con un batch: `model(**batch)` calcula salidas a partir de `input_ids` y máscaras.
- **`last_hidden_state`:** Matriz de **embeddings contextualizados**: una fila por posición de token tras la **última capa** del encoder.
- **`model.eval()`:** Modo evaluación: desactiva *dropout* y comportamiento aleatorio de entrenamiento.
- **`torch.no_grad()`:** Bloque donde **no se guardan gradientes** (ahorra memoria en inferencia).
- **`**batch`:** “Desempaqueta” el diccionario en argumentos nombrados: `input_ids=..., attention_mask=...`.

**Qué practicas:** Cargar el **cuerpo** del modelo (en BERT, solo el *encoder*) y obtener `last_hidden_state`: un **vector por cada token** en la última capa. En inferencia se usa `model.eval()` y `torch.no_grad()` para no calcular gradientes ni gastar memoria extra.

**Ejemplo mínimo:** Con `bert-base`, si hay 7 tokens, `last_hidden_state` suele tener forma `[1, 7, 768]`: 1 frase en el lote, 7 posiciones, 768 números por posición (tamaño típico “base”).

**Sintaxis:**
```python
from transformers import AutoModel
model = AutoModel.from_pretrained("bert-base-uncased")
model.eval()
with torch.no_grad():
    outputs = model(**batch)
outputs.last_hidden_state.shape
```

**Tarea:** Carga `bert-base-uncased`, pasa el `batch` del ejercicio 4 por el modelo e imprime la forma de `last_hidden_state` (batch, seq_len, hidden_size).

In [6]:
# AutoModel carga el backbone (solo encoder en BERT), sin cabeza de clasificación
from transformers import AutoModel

# Descarga/carga pesos y coloca el modelo en el mismo device que el batch
model = AutoModel.from_pretrained("bert-base-uncased").to(device)
# Modo evaluación: desactiva dropout y comportamiento de entrenamiento
model.eval()

# Sin gradientes: ahorra memoria en inferencia
with torch.no_grad():
    # **batch expande input_ids, attention_mask, etc. como argumentos nombrados
    outputs = model(**batch)

# last_hidden_state: tensor [batch, seq_len, hidden_size] (768 en bert-base)
print(outputs.last_hidden_state.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 7, 768])


---
## Ejercicio 6 — Extraer el vector del token `[CLS]`

**Antes: qué significa cada cosa**

- **`[CLS]`:** Token especial de **clasificación** al inicio de la secuencia en BERT; su vector se entrena a menudo para resumir la frase en tareas posteriores.
- **Índice `0` en la dimensión de secuencia:** Primera posición temporal = sitio de `[CLS]` cuando solo hay una oración.
- **Slice `[:, 0, :]`:** En un tensor 3D `[lote, posición, dimensión]`, conserva **todas las filas del lote**, solo la **posición 0**, y todas las dimensiones del embedding.
- **Norma L2 (`norm`):** Longitud del vector en el espacio euclídeo; un número único que resume la “magnitud” del embedding.

**Qué practicas:** En BERT, el **primer token** de la secuencia suele ser el especial `[CLS]`. Su fila en `last_hidden_state` se usa mucho como **resumen de toda la frase** (sobre todo después de *fine-tuning* o en pruebas rápidas).

**Ejemplo mínimo:** Si `last_hidden_state` tiene forma `[1, 7, 768]`, entonces `[:, 0, :]` te deja `[1, 768]`: un solo vector por cada texto del lote.

**Sintaxis:**
```python
cls_vec = outputs.last_hidden_state[:, 0, :]  # [batch, hidden]
```

**Tarea:** Calcula `cls_vec` para el batch actual e imprime su forma y la norma L2 del vector (`.norm()`).

In [7]:
# Índice 0 en la dimensión de secuencia = primer token = [CLS] en una sola frase
cls_vec = outputs.last_hidden_state[:, 0, :]
# Debe ser [batch, 768] con bert-base
print("shape:", cls_vec.shape)
# Norma euclídea del vector (magnitud; útil para comparar embeddings)
print("norm L2:", cls_vec.norm().item())

shape: torch.Size([1, 768])
norm L2: 14.117467880249023


---
## Ejercicio 7 — `pipeline` de análisis de sentimientos

**Antes: qué significa cada cosa**

- **`pipeline("sentiment-analysis")`:** Configura una tubería para **clasificar polaridad** (positivo/negativo u otras etiquetas del modelo).
- **`label`:** Nombre de la clase predicha (p. ej. `POSITIVE`), según cómo esté entrenado el modelo.
- **`score`:** Número entre 0 y 1 asociado a la confianza en esa etiqueta (no siempre es una probabilidad calibrada, pero sirve para comparar).
- **Modelo por defecto:** Si no pasas `model=...`, Hugging Face elige uno estándar (suele ser **inglés**).

**Qué practicas:** Usar **`pipeline`**: un atajo de alto nivel que junta **tokenizador + modelo + pasos finales** (por ejemplo, softmax y etiqueta) para una **tarea con nombre**. Así pruebas algo útil en pocas líneas, sin montar tú el bucle de inferencia.

**Ejemplo mínimo:** Una salida típica es una lista con un diccionario: `[{'label': 'POSITIVE', 'score': 0.99}]` → etiqueta predicha y confianza aproximada. El modelo por defecto suele estar en **inglés**.

**Sintaxis:**
```python
from transformers import pipeline
clf = pipeline("sentiment-analysis")
clf("I love this library!")
```

**Tarea:** Crea un pipeline `sentiment-analysis` (modelo por defecto está en inglés). Clasifica las frases `"This is great!"` y `"This is terrible."` e imprime las etiquetas y puntuaciones.

In [8]:
# pipeline encapsula tokenizador + modelo + postproceso para una tarea concreta
from transformers import pipeline

# Si no indicas model=..., descarga un modelo por defecto (suele ser inglés SST-2)
clf = pipeline("sentiment-analysis")
# Cada llamada devuelve una lista de dicts con "label" y "score" (probabilidad)
for frase in ["This is great!", "This is terrible."]:
    print(clf(frase))

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9998694658279419}]
[{'label': 'NEGATIVE', 'score': 0.9996345043182373}]


---
## Ejercicio 8 — `pipeline` de máscara (`fill-mask`)

**Antes: qué significa cada cosa**

- **`fill-mask`:** Tipo de `pipeline` que **rellena un único hueco** `[MASK]` con las palabras más probables según el contexto.
- **MLM (Masked Language Modeling):** Objetivo de entrenamiento: el modelo aprende a **predecir tokens ocultos** en una frase.
- **`[MASK]`:** Token reservado que marca **dónde** falta la palabra a predecir.
- **`top_k`:** Cuántas **mejores opciones** quieres ver (no solo la primera).
- **`token_str`:** Candidato como **texto**; **`score`:** puntuación relativa de esa opción (cuanto mayor, más plausible según el modelo).

**Qué practicas:** Completar un hueco marcado con **`[MASK]`** usando un modelo entrenado con **MLM** (*masked language modeling*): aprendió a adivinar palabras ocultas en contexto. Por eso necesitas un modelo tipo BERT/DistilBERT, no uno solo de generación tipo GPT puro sin cabeza MLM.

**Ejemplo mínimo:** En `"The sky is [MASK]."` el modelo sugiere palabras plausibles (`blue`, `clear`, …). **Importante:** eso es “qué palabra encaja estadísticamente”, **no** una base de datos de hechos: en frases de geografía puede equivocarse.

**Sintaxis:**
```python
unmask = pipeline("fill-mask", model="distilbert-base-uncased")
unmask("The capital of France is [MASK].")
```

**Tarea:** Crea el pipeline anterior y muestra las 3 mejores predicciones para la frase indicada (cada resultado suele traer `token_str` y `score`).

In [9]:
# fill-mask requiere un modelo entrenado con objetivo MLM (BERT/DistilBERT, etc.)
unmask = pipeline("fill-mask", model="distilbert-base-uncased")
# [MASK] es el hueco que el modelo intenta rellenar; top_k limita cuántas opciones ves
resultados = unmask("The capital of France is [MASK].", top_k=3)
# Cada r incluye el token candidato y su probabilidad relativa
for r in resultados:
    print(r["token_str"], round(r["score"], 4))

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

marseille 0.1427
nantes 0.0902
toulouse 0.0881


---
## Ejercicio 9 — Lote de frases: `padding` y `truncation`

**Antes: qué significa cada cosa**

- **Lote (*batch*):** Varios textos procesados **juntos** en la misma llamada (matriz con una fila por texto).
- **`padding`:** Rellenar con un token especial (**PAD**) hasta igualar la **longitud de la frase más larga** del lote (o hasta `max_length`).
- **`truncation`:** **Cortar** tokens sobrantes si la frase supera el límite (`max_length`).
- **`max_length`:** Número **máximo** de tokens permitidos para cada secuencia tras truncar.
- **`attention_mask`:** Indica qué posiciones son **reales** (`1`) y cuáles son **relleno** (`0`) para que la atención las ignore.

**Qué practicas:** Meter **varias frases a la vez** en una sola matriz. Como tienen distinta longitud en tokens, hace falta **igualar filas**: `padding=True` añade un token de relleno al final de las cortas; `truncation=True` **corta** las que pasan de `max_length`. La **`attention_mask`** distingue: `1` = token real, `0` = relleno (el modelo no debe “atender” al padding).

**Ejemplo mínimo:** Frase A = 4 tokens, frase B = 12 tokens. Tras el padding, ambas filas tienen la misma longitud; la fila de A llevará varios `0` al final en la máscara donde solo hay relleno.

**Sintaxis:**
```python
textos = ["Short.", "A much longer sentence here."]
batch = tokenizer(
    textos,
    padding=True,
    truncation=True,
    max_length=16,
    return_tensors="pt",
)
```

**Tarea:** Tokeniza dos frases de distinta longitud con `max_length=16` y muestra `attention_mask` para ver qué posiciones son reales (1) frente a padding (0).

In [10]:
# Dos frases: la segunda es más larga y puede cortarse con max_length
textos = ["Short.", "A much longer sentence here that might get truncated."]
# padding=True alinea longitudes; truncation recorta; max_length fija el tope de tokens
batch_multi = tokenizer(
    textos,
    padding=True,
    truncation=True,
    max_length=16,
    return_tensors="pt",
)
# 1 = token real, 0 = padding (filas con distinta cantidad de unos si hay relleno)
print("attention_mask:\n", batch_multi["attention_mask"])

attention_mask:
 tensor([[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])


---
## Ejercicio 10 — De IDs a texto: `convert_ids_to_tokens` y `decode`

**Antes: qué significa cada cosa**

- **`convert_ids_to_tokens`:** Traduce cada entero de `input_ids` a su **pieza** (subpalabra o símbolo), incluyendo prefijos `##` en WordPiece/BPE cuando un trozo continúa la palabra anterior.
- **`decode`:** Junta los IDs en una **cadena legible**; puede unir subpalabras y gestionar espacios; opcionalmente **omite tokens especiales**.
- **Tokens especiales:** Marcadores como `[CLS]`, `[SEP]` y el de **relleno** (suele llamarse `[PAD]` en BERT) con funciones fijas (inicio, fin, padding).
- **Subpalabras:** Partes de palabra (`playing` → `play` + `##ing`) para cubrir vocabulario grande con pocos símbolos base.

**Qué practicas:** **Inspeccionar al revés** el resultado del tokenizador: de lista de enteros (`input_ids`) a **trozos legibles** (`convert_ids_to_tokens`, con subpalabras tipo `##ing` si aplica) y a una **cadena reunida** (`decode`). Así entiendes por qué una palabra se partió en varios tokens.

**Ejemplo mínimo:** Puedes ver algo como `['[CLS]', 'hello', ',', 'world', '!', '[SEP]']`. Con `decode(ids)` obtienes una sola cadena; según opciones (`skip_special_tokens=True/False`) verás u ocultarás `[CLS]` y `[SEP]`.

**Sintaxis:**
```python
ids = tokenizer("Hello, world!")["input_ids"]
tokenizer.convert_ids_to_tokens(ids)
tokenizer.decode(ids)
```

**Tarea:** Para el texto `"Hello, world!"`, imprime la lista de tokens y el string decodificado.

In [11]:
# input_ids: lista de enteros que el tokenizador asignó a la frase
ids = tokenizer("Hello, world!")["input_ids"]
# Piezas de subpalabra y tokens especiales ([CLS], [SEP]) legibles
print("tokens:", tokenizer.convert_ids_to_tokens(ids))
# Reconstruye texto; por defecto puede incluir especiales (ver skip_special_tokens)
print("decode:", tokenizer.decode(ids))

tokens: ['[CLS]', 'hello', ',', 'world', '!', '[SEP]']
decode: [CLS] hello, world! [SEP]


---
## Cierre

**Qué significa lo que viene después de estos ejercicios**

- **`AutoModelFor...`:** Variante de `AutoModel` que ya incluye una **cabeza** para una tarea: por ejemplo `AutoModelForSequenceClassification` añade una capa que convierte el vector `[CLS]` en **logits por clase** (sentimiento, tópicos, etc.).
- **Hub de Hugging Face:** Repositorio web donde se publican **modelos y tokenizadores** listos para descargar con `from_pretrained`.

Has recorrido el flujo típico: **tokenizador → tensores → `AutoModel` → tensores de salida**, más **pipelines** de alto nivel y utilidades de **decodificación**. Para ir más allá, prueba otros `AutoModelFor*` (`...ForSequenceClassification`, etc.) y el [Hub de Hugging Face](https://huggingface.co/models).